In [4]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!java --version

openjdk 11.0.30 2026-01-20
OpenJDK Runtime Environment Temurin-11.0.30+7 (build 11.0.30+7)
OpenJDK 64-Bit Server VM Temurin-11.0.30+7 (build 11.0.30+7, mixed mode)


In [6]:
import os
import sys

jdk_home = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"
python_executable = sys.executable
os.environ["JAVA_HOME"] = jdk_home
os.environ["PATH"] = jdk_home + r"\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = python_executable
os.environ["PYSPARK_DRIVER_PYTHON"] = python_executable


In [7]:
import os
import sys
jdk_home = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"
python_executable = sys.executable
os.environ["JAVA_HOME"] = jdk_home
os.environ["PATH"] = jdk_home + r"\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = python_executable
os.environ["PYSPARK_DRIVER_PYTHON"] = python_executable
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ComparisonDemo").getOrCreate()



In [ ]:
!java -version

java version "24.0.2" 2025-07-15
Java(TM) SE Runtime Environment (build 24.0.2+12-54)
Java HotSpot(TM) 64-Bit Server VM (build 24.0.2+12-54, mixed mode, sharing)


In [8]:
import pandas as pd
import time
import random
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.functions import sum

In [9]:
print("Pandas Processing: ")
print()
start = time.time()
df = pd.read_csv("netflix_titles.csv")
result = df.groupby("type").size()
print(result)
end = time.time()
print("Pandas Execution Time: ", end - start)

Pandas Processing: 

type
Movie      6131
TV Show    2676
dtype: int64
Pandas Execution Time:  0.1607203483581543


In [10]:
print("Dataset info: ")
print()
print(df.shape)
print(df.info())

Dataset info: 

(8807, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB
None


In [11]:
df["type"].value_counts()

type
Movie      6131
TV Show    2676
Name: count, dtype: int64

In [12]:
print("PySpark Processing: ")
print()
spark = SparkSession.builder.appName("ComparisonDemo").getOrCreate()
start = time.time()
spark_df = spark.read.csv("netflix_titles.csv", header = True, inferSchema = True)
spark_df.groupBy("type").count().show()
end = time.time()
print("PySpark Execution Time: ", end - start)
print("Number of partitions: ", spark_df.rdd.getNumPartitions())

PySpark Processing: 

+-------------+-----+
|         type|count|
+-------------+-----+
|         NULL|    1|
|      TV Show| 2676|
|        Movie| 6131|
|William Wyler|    1|
+-------------+-----+

PySpark Execution Time:  11.683828115463257
Number of partitions:  1


In [13]:
spark = SparkSession.builder.master("local[*]").appName("SmokeTest").getOrCreate()
print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Spark version: 3.5.1
Default parallelism: 8


In [14]:
spark = SparkSession.builder.appName("Netflix Data Analysis Demo").getOrCreate()
df = spark.read.csv("netflix_titles.csv", header = True, inferSchema = True)
print("Dataset loaded successfully!")

Dataset loaded successfully!


In [15]:
df.show(5)

+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|Julien Leclercq|Sami Bouajila, Tr...|         NULL|Septem

In [16]:
print("Dataset Schema")
df.printSchema()

Dataset Schema
root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [17]:
print("Total records: ", df.count())

Total records:  8809


In [18]:
print("Number of partitions: ", df.rdd.getNumPartitions())

Number of partitions:  1


In [19]:
print("Movies vs TV Shows Count")
df.groupBy("type").count().show()

Movies vs TV Shows Count
+-------------+-----+
|         type|count|
+-------------+-----+
|         NULL|    1|
|      TV Show| 2676|
|        Movie| 6131|
|William Wyler|    1|
+-------------+-----+



In [20]:
print("Movies after 2015")
df.filter(col("release_year") > 2015).show(10)

Movies after 2015
+-------+-------+--------------------+--------------------+--------------------+--------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|       country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL| United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|  South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|     Juli

In [ ]:
print("Top Countries")
df.groupBy("country").count().orderBy("count", ascending = False).show(10)

Top Countries
+--------------+-----+
|       country|count|
+--------------+-----+
| United States| 2805|
|         India|  972|
|          NULL|  832|
|United Kingdom|  419|
|         Japan|  245|
|   South Korea|  199|
|        Canada|  181|
|         Spain|  145|
|        France|  123|
|        Mexico|  110|
+--------------+-----+
only showing top 10 rows


In [ ]:
print("Genre Distribution")
df.groupBy("listed_in").count().show(10)
spark.stop()

Genre Distribution
+--------------------+-----+
|           listed_in|count|
+--------------------+-----+
|TV Action & Adven...|    2|
|Kids' TV, TV Come...|    1|
|Dramas, Faith & S...|    1|
|Anime Series, Cri...|    1|
|Dramas, Internati...|   25|
|Reality TV, TV Co...|    2|
|              71 min|    1|
|TV Action & Adven...|    1|
|   Janeane Garofalo"|    1|
|Classic & Cult TV...|    2|
+--------------------+-----+
only showing top 10 rows
